# DDI GNN training on a free GPU (Colab or Kaggle)

Clones the repo, rebuilds the cleaned data and graph cache from the raw DrugBank file,
checks they match the committed `results/cleaning_log.json`, then runs `src/train.py`.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Checkpoints and logs go to Google Drive
  (`MyDrive/ddi_project_runs`), so a disconnect loses at most one epoch: just re-run all cells.
- **Kaggle:** Settings → Accelerator **GPU T4**, and turn **Internet on**. Outputs go to
  `/kaggle/working/ddi_runs`. For long runs use *Save Version → Save & Run All* so it keeps going
  after you close the tab. To continue runs started elsewhere, upload that output folder (e.g. the
  Colab `ddi_project_runs` Drive folder, as a zip) as a Kaggle dataset and add it as an input.
- Set `REPO_URL` below. For a private repo, add a secret named `GITHUB_TOKEN` (Colab: key icon in
  the sidebar; Kaggle: Add-ons → Secrets) holding a GitHub token with read access.

Every run uses `--resume`: finished runs are skipped, interrupted ones continue from their last epoch.

In [ ]:
REPO_URL = "https://github.com/KritiWadhawan/ddi-project.git"
BRANCH = "master"

# One entry per training run, in order. Arguments go straight to src/train.py.
#
# Round 1 (done): S1 screening of class weighting x regularization, seeds 0-2.
# Round 1b (done): fingerprint fusion and/or half depth. Winner: 4 layers + --fp.
# Round 1c (done): graph augmentation (--drop-edge, --mask-atom) and dropping class
# weights entirely. All four variants LOST on S1 test on all three seeds, while final
# training loss rose 0.04 -> ~0.12 (memorisation cut ~3x) with no validation movement.
# The S1 ceiling is drug coverage (~1,023 training drugs), not overfitting.
# Round 2 (done): the winner across all three evaluation regimes, 5 seeds each for
# S1/S2. Test macro_f1_occurring, GNN vs the Morgan+RF baseline:
#     transductive  0.950 vs 0.808   (+0.142)
#     S2            0.585 vs 0.594   (-0.009, RF wins 4 of 5 seeds)
#     S1            0.314 vs 0.270   (+0.044 vs plain RF; +0.006 vs the
#                                     prior-adjusted RF, i.e. parity)
# Unseeing ONE drug costs the GNN 38% of its transductive score and the fingerprint
# forest only 26%: the GNN's transductive lead is drug identity it has memorised, and
# that is the first thing lost. This monotone collapse is the project's main result.
REG = "--gnn-dropout 0.3 --adamw --weight-decay 1e-2 --lr 5e-4"
BASE = f"--class-weights sqrt {REG}"
WINNER = f"{BASE} --layers 4 --fp"   # unchanged since round 1b

# Round 3 (current): the architecture ablations, on TRANSDUCTIVE.
# They were run on S1 in round 2 and established nothing. On S1 seed 0 validation --
# the only basis selection may use -- mean pooling scored 0.2811 against co-attention's
# 0.2803, so the borrowed co-attention readout bought nothing, and the apparent +0.024
# on test is one seed inside a regime whose seed spread is +/-0.043. That is the
# expected outcome: round 1c showed S1 is coverage-limited, so encoder architecture
# cannot show up there. Transductive is where the GNN wins by 0.142, so it is where the
# question "does the architecture earn that win?" is actually answerable.
RUNS = [
    f"--split transductive --seed 0 {WINNER} --readout mean",   # co-attention -> mean pooling
    f"--split transductive --seed 0 {WINNER} --no-inter",       # drop the bipartite GATConv
]
# 2 runs. The completed transductive run took 63 epochs at ~60 s each, so budget ~1.2 h
# per ablation, ~2.5 h in total. --resume skips whatever is already finished, and
# neither name collides with the round-2 run (transductive_seed0_coattn_L4_...).
#
# Optional third ablation, and the one that tests the collapse claim most directly:
# drop the graph encoder's contribution by removing --fp, leaving the GNN alone, or
# conversely check how far the fingerprint alone gets in this regime. Uncomment to add.
# RUNS += [f"--split transductive --seed 0 {BASE} --layers 4"]   # no fingerprint fusion

COMMON_ARGS = "--batch-size 256 --workers 2 --resume"

## Environment, GPU check, persistent output folder

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other")
!nvidia-smi --query-gpu=name,memory.total --format=csv

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/ddi_project_runs")
elif ON_KAGGLE:
    OUT = Path("/kaggle/working/ddi_runs")
else:
    OUT = Path.home() / "ddi_project_runs"
OUT.mkdir(parents=True, exist_ok=True)
if ON_KAGGLE and not (OUT / "gnn_metrics.csv").exists():
    # Carry over earlier runs (e.g. the Colab Drive folder uploaded as a Kaggle dataset)
    # so finished runs are skipped and interrupted ones resume.
    import shutil
    for prev in Path("/kaggle/input").rglob("gnn_metrics.csv"):
        shutil.copytree(prev.parent, OUT, dirs_exist_ok=True)
        print("carried over earlier runs from", prev.parent)
        break
REPO = Path("/tmp/ddi-project")
print("outputs ->", OUT)

## Clone the repo

In [ ]:
def secret(name):
    try:
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1

## Install dependencies

Colab and Kaggle already have CUDA PyTorch, pandas, pyarrow and scikit-learn. Only PyG and RDKit
are added, pinned to the versions used locally so featurization matches.

In [ ]:
!pip install -q torch_geometric==2.8.1 rdkit==2026.3.6
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: switch the runtime/accelerator to a GPU first"

## Link outputs to the persistent folder

`checkpoints/`, `results/runs/` and `results/gnn_metrics.csv` live in `OUT`, so they survive a
disconnect and `--resume` finds them.

In [ ]:
import shutil

(OUT / "checkpoints").mkdir(exist_ok=True)
(OUT / "runs").mkdir(exist_ok=True)
for link, target in [
    (REPO / "checkpoints", OUT / "checkpoints"),
    (REPO / "results" / "runs", OUT / "runs"),
    (REPO / "results" / "gnn_metrics.csv", OUT / "gnn_metrics.csv"),
]:
    if link.is_symlink():
        continue
    if link.exists():
        # Results committed to the repo come down with the clone as real files. Seed OUT
        # with any it is missing (so those runs count as finished and get skipped), keep
        # OUT's own copy otherwise, then replace the real path with the symlink.
        for src in sorted(link.iterdir()) if link.is_dir() else [link]:
            dest = target / src.name if link.is_dir() else target
            if not dest.exists():
                shutil.copy2(src, dest)
        shutil.rmtree(link) if link.is_dir() else link.unlink()
    link.symlink_to(target)
!ls -la checkpoints results/runs results/gnn_metrics.csv

## Rebuild the data

Downloads the raw DrugBank DDI file (cached in `OUT`), then runs the same scripts as locally.
The cleaned counts must match the committed `results/cleaning_log.json`.

In [ ]:
import json, shutil

expected = json.loads(Path("results/cleaning_log.json").read_text())

raw_cache = OUT / "drugbank_ddi.tab"
if not raw_cache.exists():
    !wget -q -O {raw_cache} https://dataverse.harvard.edu/api/access/datafile/4139573
Path("data").mkdir(exist_ok=True)
shutil.copy(raw_cache, "data/drugbank_ddi.tab")

!python src/convert_to_parquet.py
!python src/clean.py > /dev/null
!python src/graphs.py

got = json.loads(Path("results/cleaning_log.json").read_text())
for key in ["raw_rows", "conflicting_pairs", "clean_rows", "clean_drugs", "clean_classes"]:
    assert got[key] == expected[key], f"{key}: got {got[key]}, committed {expected[key]}"
print("cleaned data matches the committed cleaning log")

## Speed check

Times a few training steps of the full model without writing anything, and projects the epoch time.

In [ ]:
import time
import torch.nn.functional as F
sys.path.insert(0, str(REPO / "src"))
from graphs import load_graphs
from splits import load_data, make_split
from model import DDIModel
from train import PairLoader

train_df, _, _ = make_split("S1", load_data(), 0)
loader = PairLoader(train_df, load_graphs(), 256, shuffle=True)
model = DDIModel().cuda()
opt = torch.optim.Adam(model.parameters())
times = []
for i, (a, b, y) in zip(range(6), loader):
    t0 = time.time()
    loss = F.cross_entropy(model(a.cuda(), b.cuda()), y.cuda())
    opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    times.append(time.time() - t0)
per_batch = sum(times[1:]) / len(times[1:])
print(f"{per_batch:.2f} s/batch (GPU step only) -> ~{per_batch * len(loader) / 60:.1f} min per S1 epoch + validation")
print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
del model, opt
torch.cuda.empty_cache()

## One-off: archive the superseded S1 runs

Round 2 re-runs S1 seeds 0-2 under the new checkpoint-selection rule. Their round-1b
rows and checkpoints are moved aside, not deleted, so `--resume` sees those three runs
as unfinished and nothing is lost.

In [ ]:
import csv
from pathlib import Path

STALE = [f"S1_seed{s}_coattn_L4_cwsqrt_do0.3_adamw_lr0.0005_wd0.01_fp" for s in (0, 1, 2)]
csv_path = Path("results/gnn_metrics.csv").resolve()   # follow the symlink into Drive
archive = csv_path.with_name("gnn_metrics_superseded.csv")
marker = Path("checkpoints") / f"{STALE[0]}.pt.round1b"

if marker.exists():
    print("already archived; nothing to do")
else:
    rows = list(csv.reader(open(csv_path, newline="")))
    header, body = rows[0], rows[1:]
    keep = [r for r in body if r[2] not in STALE]
    moved = [r for r in body if r[2] in STALE]
    if moved:
        write_header = not archive.exists()
        with open(archive, "a", newline="") as f:
            w = csv.writer(f)
            if write_header:
                w.writerow(header)
            w.writerows(moved)
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(header)
            w.writerows(keep)
    for run in STALE:
        for rel in (f"checkpoints/{run}.pt", f"checkpoints/{run}_last.pt",
                    f"results/runs/{run}.csv"):
            p = Path(rel)
            if p.exists():
                p.rename(p.with_name(p.name + ".round1b"))
    print(f"archived {len(moved)} rows to {archive.name}; {len(keep)} rows remain")

## Train

Runs everything in `RUNS` in order. Safe to re-run this cell after a disconnect: finished runs are
skipped and interrupted ones resume. The stop button ends the whole loop, not just the current run.

In [ ]:
import shlex

for run_args in RUNS:
    print("=" * 80, f"\n{run_args}\n" + "=" * 80, flush=True)
    cmd = [sys.executable, "-u", "-W", "ignore", "src/train.py", *shlex.split(run_args), *shlex.split(COMMON_ARGS)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("\nStopped. Re-run this cell later to resume where it left off.")
        break
    if proc.wait() != 0:
        raise RuntimeError(f"train.py exited with code {proc.returncode} for: {run_args}")

## Results so far (vs. RF+Morgan baseline)

In [ ]:
import pandas as pd

# macro_f1 is the OLD column (sklearn's default label set: classes in y_true OR y_pred).
# It is not comparable between models -- a model that predicts fewer rare classes removes
# its own zero-F1 entries and scores higher. See results/logit_adjustment.md.
PRIMARY = "macro_f1_occurring"

gnn = pd.read_csv("results/gnn_metrics.csv")
gnn["variant"] = gnn["model"].str.replace(r"^(S1|S2|transductive)_seed\d+_", "", regex=True)
summary = (gnn.groupby(["split", "variant", "set"])[["acc", PRIMARY, "macro_f1"]]
              .agg(["mean", "std", "count"]).round(3))
display(summary)

# Selection: S1 seeds 0-2 only, so every variant is compared on the same seeds, and on
# VALIDATION only, never test. Runs recorded before the metric change have PRIMARY blank
# until src/rescore.py refills them from their checkpoints -- those appear as NaN here
# rather than silently showing the old, non-comparable number.
s1 = gnn[(gnn.split == "S1") & gnn.seed.isin([0, 1, 2]) & (gnn.set == "val")]
pick = s1.groupby("variant")[PRIMARY].agg(["mean", "std", "count"]).round(3)
display(pick.sort_values("mean", ascending=False).rename(columns={"mean": "val_mean"}))

# The bar on S1 test is the RF baseline WITH its own validation-tuned prior adjustment
# (0.322), not the plain RF (0.289): the same correction was offered to the GNN and did
# nothing for it, so reporting only the unadjusted baseline would flatter us.
rf = pd.read_csv("results/baseline_metrics.csv")
rf = rf[(rf.split == "S1") & rf.seed.isin([0, 1, 2]) & (rf.model == "RF+Morgan")]
display(rf.groupby("set")[[PRIMARY, "macro_f1"]].agg(["mean", "std"]).round(3))
print("target: RF + validation-tuned prior adjustment, S1 test macro_f1_occurring = 0.322")

## Bring results back to the local repo

Download this zip and unpack it into the local repo's `results/` folder to commit the logs and
metrics. Checkpoints stay in `OUT` (they are gitignored); copy them separately for Phase 4.

In [ ]:
!cd {OUT} && rm -f gnn_results.zip && zip -qr gnn_results.zip runs gnn_metrics.csv && ls -la gnn_results.zip
if ON_COLAB:
    from google.colab import files
    files.download(str(OUT / "gnn_results.zip"))